In [3]:
%pip install PuLP

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import pandas as pd
import numpy as np
from pathlib import Path
import openpyxl
import pulp as pl

# =========================================================
# 0. Basic settings
# =========================================================
PRICE_FILE = Path("Prices.xlsx")
EMISSIONS_FACTOR = 0.05306  # tonne CO2 / MMBtu
START_DATE = pd.to_datetime("2022-03-21").date()
END_DATE = pd.to_datetime("2022-03-27").date()

# =========================================================
# 1. Load input data
# =========================================================
price_electric = pd.read_excel(PRICE_FILE, sheet_name="PRICE_ELECTRIC")
price_gas = pd.read_excel(PRICE_FILE, sheet_name="PRICE_GAS")

wb = openpyxl.load_workbook(PRICE_FILE, data_only=True)
co2_sheet = wb["PRICE_CO2"]
CO2_PRICE = float(list(co2_sheet.values)[0][0])

electric_date_col = "OPERATING_DATE"
electric_hour_col = "HOUR_ENDING"
electric_price_col = "NP15 ($/MWh)"

gas_date_col = "OPERATING_DATE"
gas_price_col = "PG&E Citygate ($/MMBtu)"

elec = price_electric[[electric_date_col, electric_hour_col, electric_price_col]].copy()
elec.columns = ["OPERATING_DATE", "HOUR_ENDING", "PRICE_ELECTRIC"]

gas = price_gas[[gas_date_col, gas_price_col]].copy()
gas.columns = ["OPERATING_DATE", "PRICE_GAS"]

elec["OPERATING_DATE"] = pd.to_datetime(elec["OPERATING_DATE"]).dt.date
gas["OPERATING_DATE"] = pd.to_datetime(gas["OPERATING_DATE"]).dt.date

elec = elec[(elec["OPERATING_DATE"] >= START_DATE) & (elec["OPERATING_DATE"] <= END_DATE)].copy()
gas = gas[(gas["OPERATING_DATE"] >= START_DATE) & (gas["OPERATING_DATE"] <= END_DATE)].copy()

hourly = (
    elec.merge(gas, on="OPERATING_DATE", how="left")
        .sort_values(["OPERATING_DATE", "HOUR_ENDING"])
        .reset_index(drop=True)
)

if hourly["PRICE_GAS"].isna().any():
    raise ValueError("Some gas prices are missing after merge.")

# =========================================================
# 2. Configuration data
# =========================================================
configs = {
    1: {
        "name": "all-off",
        "min_mw": 0.0,
        "max_mw": 0.0,
        "vom": 0.0,
        "su_dollar": 0.0,
        "su_fuel": 0.0,
        "hr_points": {}
    },
    2: {
        "name": "single CT",
        "min_mw": 57.0,
        "max_mw": 190.0,
        "vom": 5.0,
        "su_dollar": 7250.0,
        "su_fuel": 220.0,
        "hr_points": {
            "Minimum load": 12.591,
            "60% load": 10.642,
            "80% load": 10.275,
            "100% load": 10.133
        }
    },
    3: {
        "name": "two CTs, no steam",
        "min_mw": 114.0,
        "max_mw": 380.0,
        "vom": 5.0,
        "su_dollar": 14500.0,
        "su_fuel": 440.0,
        "hr_points": {
            "Minimum load": 12.591,
            "60% load": 10.642,
            "80% load": 10.275,
            "100% load": 10.133
        }
    },
    4: {
        "name": "1x1",
        "min_mw": 150.0,
        "max_mw": 340.0,
        "vom": 2.5,
        "su_dollar": 16500.0,
        "su_fuel": 850.0,
        "hr_points": {
            "Minimum load": 7.695,
            "60% load": 7.358,
            "80% load": 7.149,
            "100% load": 7.048
        }
    },
    5: {
        "name": "2x1",
        "min_mw": 312.0,
        "max_mw": 610.0,
        "vom": 2.0,
        "su_dollar": 23750.0,
        "su_fuel": 1070.0,
        "hr_points": {
            "Minimum load": 7.121,
            "60% load": 6.999,
            "80% load": 6.839,
            "100% load": 6.762
        }
    }
}

point_labels = ["Minimum load", "60% load", "80% load", "100% load"]
active_configs = [2, 3, 4, 5]

# =========================================================
# 3. Build per-hour bid points
# =========================================================
T = list(range(len(hourly)))
C = list(configs.keys())

point_data = {}
point_keys = []

for t in T:
    gas_price = float(hourly.loc[t, "PRICE_GAS"])

    for c in C:
        if c == 1:
            key = (t, c, "off")
            point_keys.append(key)
            point_data[key] = {
                "mw": 0.0,
                "fuel_cost": 0.0,
                "co2_cost": 0.0,
                "vom_cost": 0.0,
                "var_cost": 0.0
            }
            continue

        for label in point_labels:
            if label == "Minimum load":
                mw = configs[c]["min_mw"]
            elif label == "60% load":
                mw = 0.60 * configs[c]["max_mw"]
            elif label == "80% load":
                mw = 0.80 * configs[c]["max_mw"]
            else:
                mw = 1.00 * configs[c]["max_mw"]

            hr = configs[c]["hr_points"][label]
            vom = configs[c]["vom"]

            fuel_cost = hr * gas_price * mw
            co2_cost = hr * EMISSIONS_FACTOR * CO2_PRICE * mw
            vom_cost = vom * mw
            var_cost = fuel_cost + co2_cost + vom_cost

            key = (t, c, label)
            point_keys.append(key)
            point_data[key] = {
                "mw": float(mw),
                "fuel_cost": float(fuel_cost),
                "co2_cost": float(co2_cost),
                "vom_cost": float(vom_cost),
                "var_cost": float(var_cost)
            }

# =========================================================
# 4. Build MILP
# =========================================================
model = pl.LpProblem("CCGT_CAISO_Optimization", pl.LpMaximize)

# Configuration variables
x = pl.LpVariable.dicts("x", [(c, t) for c in C for t in T], cat="Binary")

# Plant-level on/off and startup/shutdown
on = pl.LpVariable.dicts("on", T, cat="Binary")
start_on = pl.LpVariable.dicts("start_on", T, cat="Binary")
stop_on = pl.LpVariable.dicts("stop_on", T, cat="Binary")

# Startup into a specific active configuration
start_cfg = pl.LpVariable.dicts("start_cfg", [(c, t) for c in active_configs for t in T], cat="Binary")

# Lambda weights for convex combination of bid points
lam = pl.LpVariable.dicts("lam", point_keys, lowBound=0, upBound=1, cat="Continuous")

# Initial state: plant starts off
on_init = 0

# Exactly one configuration each hour
for t in T:
    model += pl.lpSum(x[(c, t)] for c in C) == 1

# Link plant on/off to all-off configuration
for t in T:
    model += on[t] + x[(1, t)] == 1

# Link lambda weights to active configuration
for t in T:
    for c in C:
        keys_ct = [k for k in point_keys if k[0] == t and k[1] == c]
        model += pl.lpSum(lam[k] for k in keys_ct) == x[(c, t)]

# Startup / shutdown logic at plant level
for t in T:
    if t == 0:
        model += on[t] - on_init == start_on[t] - stop_on[t]
    else:
        model += on[t] - on[t-1] == start_on[t] - stop_on[t]

# Link startup into a specific configuration
for t in T:
    for c in active_configs:
        model += start_cfg[(c, t)] <= x[(c, t)]
        model += start_cfg[(c, t)] <= start_on[t]
        model += start_cfg[(c, t)] >= x[(c, t)] + start_on[t] - 1

# Cannot jump directly from off to 2x1
for t in T:
    if t == 0:
        model += x[(5, 0)] == 0
    else:
        model += x[(5, t)] + x[(1, t-1)] <= 1

# Per-hour operating expressions
p_t = {}
fuel_cost_t = {}
co2_cost_t = {}
vom_cost_t = {}
var_cost_t = {}

for t in T:
    keys_t = [k for k in point_keys if k[0] == t]
    p_t[t] = pl.lpSum(point_data[k]["mw"] * lam[k] for k in keys_t)
    fuel_cost_t[t] = pl.lpSum(point_data[k]["fuel_cost"] * lam[k] for k in keys_t)
    co2_cost_t[t] = pl.lpSum(point_data[k]["co2_cost"] * lam[k] for k in keys_t)
    vom_cost_t[t] = pl.lpSum(point_data[k]["vom_cost"] * lam[k] for k in keys_t)
    var_cost_t[t] = pl.lpSum(point_data[k]["var_cost"] * lam[k] for k in keys_t)

startup_nonfuel_cost_t = {}
startup_fuel_cost_t = {}
startup_co2_cost_t = {}
startup_total_cost_t = {}

for t in T:
    gas_price = float(hourly.loc[t, "PRICE_GAS"])

    startup_nonfuel_cost_t[t] = pl.lpSum(
        configs[c]["su_dollar"] * start_cfg[(c, t)] for c in active_configs
    )
    startup_fuel_cost_t[t] = pl.lpSum(
        configs[c]["su_fuel"] * gas_price * start_cfg[(c, t)] for c in active_configs
    )
    startup_co2_cost_t[t] = pl.lpSum(
        configs[c]["su_fuel"] * EMISSIONS_FACTOR * CO2_PRICE * start_cfg[(c, t)] for c in active_configs
    )
    startup_total_cost_t[t] = (
        startup_nonfuel_cost_t[t] +
        startup_fuel_cost_t[t] +
        startup_co2_cost_t[t]
    )

# Objective
model += pl.lpSum(
    float(hourly.loc[t, "PRICE_ELECTRIC"]) * p_t[t]
    - var_cost_t[t]
    - startup_total_cost_t[t]
    for t in T
)

# =========================================================
# 5. Solve
# =========================================================
solver = pl.PULP_CBC_CMD(msg=False, timeLimit=120)
model.solve(solver)

status = pl.LpStatus[model.status]
print("Solve status:", status)

if status not in ["Optimal", "Not Solved", "Undefined", "Infeasible"]:
    raise ValueError(f"Unexpected solver status = {status}")

if status != "Optimal":
    print("Warning: CBC did not return Optimal within the time limit.")

# =========================================================
# 6. Extract results
# =========================================================
rows = []

for t in T:
    active_config = max(C, key=lambda c: pl.value(x[(c, t)]))
    mw_generation = float(pl.value(p_t[t]))
    fuel_cost = float(pl.value(fuel_cost_t[t]))
    co2_cost = float(pl.value(co2_cost_t[t]))
    vom_cost = float(pl.value(vom_cost_t[t]))
    startup_nonfuel_cost = float(pl.value(startup_nonfuel_cost_t[t]))
    startup_fuel_cost = float(pl.value(startup_fuel_cost_t[t]))
    startup_co2_cost = float(pl.value(startup_co2_cost_t[t]))
    startup_total_cost = float(pl.value(startup_total_cost_t[t]))
    start_indicator = int(round(pl.value(start_on[t])))

    rows.append({
        "OPERATING_DATE": hourly.loc[t, "OPERATING_DATE"],
        "HOUR_ENDING": int(hourly.loc[t, "HOUR_ENDING"]),
        "PRICE_ELECTRIC": float(hourly.loc[t, "PRICE_ELECTRIC"]),
        "PRICE_GAS": float(hourly.loc[t, "PRICE_GAS"]),
        "CONFIGURATION_ACTIVE": active_config,
        "MW_GENERATION": mw_generation,
        "FUEL_COST": fuel_cost,
        "CO2_COST": co2_cost,
        "VOM_COST": vom_cost,
        "STARTUP_NONFUEL_COST": startup_nonfuel_cost,
        "STARTUP_FUEL_COST": startup_fuel_cost,
        "STARTUP_CO2_COST": startup_co2_cost,
        "STARTUP_TOTAL_COST": startup_total_cost,
        "START_INDICATOR": start_indicator
    })

ccgt_caiso_results = pd.DataFrame(rows)

for col in [
    "PRICE_ELECTRIC", "PRICE_GAS", "MW_GENERATION", "FUEL_COST", "CO2_COST", "VOM_COST",
    "STARTUP_NONFUEL_COST", "STARTUP_FUEL_COST", "STARTUP_CO2_COST", "STARTUP_TOTAL_COST"
]:
    ccgt_caiso_results[col] = ccgt_caiso_results[col].round(4)

# =========================================================
# 7. Save CSVs
# =========================================================
ccgt_caiso_results[
    ["OPERATING_DATE", "HOUR_ENDING", "PRICE_ELECTRIC", "PRICE_GAS", "CONFIGURATION_ACTIVE", "MW_GENERATION"]
].to_csv("CCGT_CAISO.csv", index=False)

ccgt_caiso_results.to_csv("CCGT_CAISO_FULL.csv", index=False)

# =========================================================
# 8. First 12 hours
# =========================================================
first12_caiso = ccgt_caiso_results[
    ["OPERATING_DATE", "HOUR_ENDING", "PRICE_ELECTRIC", "PRICE_GAS", "CONFIGURATION_ACTIVE", "MW_GENERATION"]
].head(12)

print("\nFirst 12 hours:")
print(first12_caiso)

# =========================================================
# 9. Summary metrics
# =========================================================
INSTALLED_CAPACITY_MW = 610.0
INSTALLED_CAPACITY_KW = 610000.0
HOURS = len(ccgt_caiso_results)

total_revenue = float((ccgt_caiso_results["PRICE_ELECTRIC"] * ccgt_caiso_results["MW_GENERATION"]).sum())

variable_fuel_cost = float(ccgt_caiso_results["FUEL_COST"].sum())
startup_fuel_cost = float(ccgt_caiso_results["STARTUP_FUEL_COST"].sum())
fuel_cost_total = variable_fuel_cost + startup_fuel_cost

variable_co2_cost = float(ccgt_caiso_results["CO2_COST"].sum())
startup_co2_cost = float(ccgt_caiso_results["STARTUP_CO2_COST"].sum())
co2_cost_total = variable_co2_cost + startup_co2_cost

vom_cost = float(ccgt_caiso_results["VOM_COST"].sum())
startup_nonfuel_cost = float(ccgt_caiso_results["STARTUP_NONFUEL_COST"].sum())

total_costs = fuel_cost_total + co2_cost_total + vom_cost + startup_nonfuel_cost

gross_margin = total_revenue - total_costs
gross_margin_per_kw = gross_margin / INSTALLED_CAPACITY_KW
capacity_factor = ccgt_caiso_results["MW_GENERATION"].sum() / (INSTALLED_CAPACITY_MW * HOURS)
number_of_starts = int(ccgt_caiso_results["START_INDICATOR"].sum())

task2cvi_summary = pd.DataFrame({
    "Metric": [
        "Gross Margin ($/kW)",
        "Capacity Factor (%)",
        "Total Revenue ($)",
        "Total Costs ($)",
        "Fuel Costs ($)",
        "Number of Starts"
    ],
    "Value": [
        gross_margin_per_kw,
        capacity_factor * 100,
        total_revenue,
        total_costs,
        fuel_cost_total,
        number_of_starts
    ]
})

task2cvi_summary["Value"] = task2cvi_summary["Value"].round(4)

print("\nTask 2c(vi) summary:")
print(task2cvi_summary)

# Keep these objects in memory for later tasks
task2ciii_first12 = first12_caiso.copy()
task2cvi_summary = task2cvi_summary.copy()

Solve status: Optimal

First 12 hours:
   OPERATING_DATE  HOUR_ENDING  PRICE_ELECTRIC  PRICE_GAS  \
0      2022-03-21            1           45.04       7.05   
1      2022-03-21            2           43.63       7.05   
2      2022-03-21            3           43.43       7.05   
3      2022-03-21            4           43.42       7.05   
4      2022-03-21            5           46.30       7.05   
5      2022-03-21            6           55.50       7.05   
6      2022-03-21            7           76.70       7.05   
7      2022-03-21            8           77.29       7.05   
8      2022-03-21            9           51.19       7.05   
9      2022-03-21           10           46.89       7.05   
10     2022-03-21           11           43.54       7.05   
11     2022-03-21           12           37.18       7.05   

    CONFIGURATION_ACTIVE  MW_GENERATION  
0                      1            0.0  
1                      1            0.0  
2                      1            0.0  